In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("screen_time_behavior.csv")

: 

## Data Cleaning and Missing Value Imputation

In [ ]:
print("Checking for missing values:")
print(data.isnull().sum())

# Impute missing values for numeric columns with the mean
# Identify numeric columns
numeric_cols = data.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if data[col].isnull().any():
        mean_value = data[col].mean()
        data[col].fillna(mean_value, inplace=True)
        print(f"Filled missing values in '{col}' with mean: {mean_value:.2f}")

print("\nAfter imputation, missing values:")
print(data.isnull().sum())

## 1. Central Tendency of Data (Visual and Numeric)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Choose a numeric column for analysis
numeric_column = 'weekday_screen_hours'

# Numeric Summary
mean_val = data[numeric_column].mean()
median_val = data[numeric_column].median()
mode_val = data[numeric_column].mode()[0] # mode() can return multiple values, take the first one

print(f"Central Tendency for '{numeric_column}':")
print(f"  Mean: {mean_val:.2f}")
print(f"  Median: {median_val:.2f}")
print(f"  Mode: {mode_val:.2f}")

# Visual Summary (Histogram with KDE and central tendency markers)
plt.figure(figsize=(10, 6))
sns.histplot(data[numeric_column], kde=True, bins=30, color='skyblue')
plt.axvline(mean_val, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_val:.2f}')
plt.axvline(median_val, color='green', linestyle='dashed', linewidth=2, label=f'Median: {median_val:.2f}')
plt.axvline(mode_val, color='purple', linestyle='dashed', linewidth=2, label=f'Mode: {mode_val:.2f}')
plt.title(f'Distribution and Central Tendency of {numeric_column}')
plt.xlabel(numeric_column)
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 2. Outlier Identification (Visual and Numeric)

In [ ]:
# Choose a numeric column for analysis (re-defining for clarity, though it's set above)
numeric_column = 'weekday_screen_hours'

# Visual Outlier Identification (Box Plot)
plt.figure(figsize=(10, 6))
sns.boxplot(x=data[numeric_column], color='lightgreen')
plt.title(f'Box Plot of {numeric_column} for Outlier Identification')
plt.xlabel(numeric_column)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Numeric Outlier Identification (IQR method)
Q1 = data[numeric_column].quantile(0.25)
Q3 = data[numeric_column].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data[numeric_column] < lower_bound) | (data[numeric_column] > upper_bound)]

print(f"Numeric Outlier Identification for '{numeric_column}' (IQR Method):")
print(f"  Q1: {Q1:.2f}")
print(f"  Q3: {Q3:.2f}")
print(f"  IQR: {IQR:.2f}")
print(f"  Lower Bound: {lower_bound:.2f}")
print(f"  Upper Bound: {upper_bound:.2f}")

if not outliers.empty:
    print(f"\nIdentified {len(outliers)} outliers:")
    display(outliers[[numeric_column]].head())
else:
    print("\nNo significant outliers found using the IQR method. Adding artificial outliers for demonstration.")
    # Add artificial outliers if none exist
    data_with_outliers = data.copy()
    num_rows = len(data_with_outliers)

    # Add a few very high and very low values
    num_artificial_outliers = 5
    if num_rows > num_artificial_outliers:
        # Ensure numeric_column is actually numeric before multiplication/division
        if pd.api.types.is_numeric_dtype(data_with_outliers[numeric_column]):
            max_val = data_with_outliers[numeric_column].max()
            min_val = data_with_outliers[numeric_column].min()
            outlier_indices = np.random.choice(data_with_outliers.index, num_artificial_outliers, replace=False)
            data_with_outliers.loc[outlier_indices[:num_artificial_outliers//2], numeric_column] = max_val * 5
            data_with_outliers.loc[outlier_indices[num_artificial_outliers//2:], numeric_column] = min_val / 5
        else:
            print(f"Skipping artificial outlier generation for {numeric_column} as it is not numeric.")
    else:
        print("Dataset too small to add artificial outliers effectively.")

    plt.figure(figsize=(10, 6))
    sns.boxplot(x=data_with_outliers[numeric_column], color='lightcoral')
    plt.title(f'Box Plot of {numeric_column} with Artificial Outliers')
    plt.xlabel(numeric_column)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()
    print("Artificial outliers added and visualized.")

## 3. Visualizing Relationship Between Two Numeric Variables

In [ ]:
numeric_var1 = 'weekday_screen_hours'
numeric_var2 = 'physical_activity_hours_weekly'

plt.figure(figsize=(10, 7))
sns.scatterplot(x=numeric_var1, y=numeric_var2, data=data, alpha=0.6, hue='age_group')
plt.title(f'Scatter Plot of {numeric_var1} vs {numeric_var2}')
plt.xlabel(numeric_var1)
plt.ylabel(numeric_var2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 4. Visualizing Relationship Between Numeric and Nominal Variable

In [ ]:
numeric_var = 'weekday_screen_hours'
nominal_var = 'gender'

plt.figure(figsize=(10, 6))
sns.boxplot(x=nominal_var, y=numeric_var, data=data, palette='viridis')
plt.title(f'Distribution of {numeric_var} by {nominal_var}')
plt.xlabel(nominal_var)
plt.ylabel(numeric_var)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 5. Visualizing Relationship Between Two Nominal Variables

In [ ]:
nominal_var1 = 'gender'
nominal_var2 = 'platform'

# Create a contingency table
contingency_table = pd.crosstab(data[nominal_var1], data[nominal_var2])

print(f"Contingency Table for {nominal_var1} vs {nominal_var2}:")
display(contingency_table)

# Visualize with a stacked bar chart
contingency_table.plot(kind='bar', stacked=True, figsize=(12, 7), cmap='Paired')
plt.title(f'Stacked Bar Chart of {nominal_var1} vs {nominal_var2}')
plt.xlabel(nominal_var1)
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title=nominal_var2)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Alternatively, using a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='Blues', linewidths=.5)
plt.title(f'Heatmap of {nominal_var1} vs {nominal_var2}')
plt.xlabel(nominal_var2)
plt.ylabel(nominal_var1)
plt.show()